# 第 13 章　分析手順と再現性

共通の集計を関数にまとめ，入力を替えながら比較を進めます。

このノートの前半は，本文の説明と一緒に実行します。各場面で先に短い問いを読み，講師の案内に合わせてセルを動かし，出力の見所を確認します。後半は約20分の実習です。前半のコードは動く見本として用意してあり，作図コードをすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path)
hourly["date"] = pd.to_datetime(hourly["dteday"])
print("Python:", sys.version.split()[0], "pandas:", pd.__version__)
hourly[["date", "hr", "cnt", "registered", "casual"]].head(3)

## 本文と一緒に実行

### 1. 二つの6月を同じ規則で集計する

2011年と2012年の6月で，朝と夕方の平均はどのくらい違うでしょうか。まず対象の行を選び，8時の一点がどの30行からできるか確かめます。

In [ ]:
june_2011 = hourly.loc[
    (hourly["date"].dt.year == 2011) & (hourly["date"].dt.month == 6)
]
june_2012 = hourly.loc[
    (hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 6)
]
print(len(june_2011), len(june_2012))

In [ ]:
plt.rcParams["figure.figsize"] = (6.6, 3.7)
fig, ax = plt.subplots(layout="constrained")
for records, label, style in [
    (june_2011, "June 2011", "o-"), (june_2012, "June 2012", "s--")
]:
    means = records.groupby("hr")["cnt"].mean()
    ax.plot(means.index, means, style, markersize=3, label=label)
ax.set(xlabel="Clock hour", ylabel="Mean rental count", xticks=range(0, 24, 3), ylim=(0, 750))
ax.legend(frameon=False)
plt.show()

In [ ]:
eight_in_june = june_2012.loc[june_2012["hr"] == 8, "cnt"]
print("Recorded hours:", len(eight_in_june))
print("Total rentals:", eight_in_june.sum())
print("Mean rentals:", eight_in_june.mean())

In [ ]:
grouped = june_2012.groupby("hr")["cnt"]
june_summary = grouped.agg(recorded_hours="size", total="sum", mean="mean")
june_summary.loc[[8, 13, 17]].round(2)

**出力の見所：**2012年6月8時は合計15,252件，30行，平均508.4件です。次は，取り替えたい入力表と列名をパラメーターにします。

### 2. 表を返す関数にする

定義セルを実行しただけでは，まだ特定の月を集計しません。呼出の二つの引数が，`records` と `value_column` のどちらに入るかを読みます。

In [ ]:
def hourly_summary(records, value_column):
    grouped = records.groupby("hr")[value_column]
    summary = grouped.agg(recorded_hours="size", total="sum", mean="mean")
    return summary

In [ ]:
profile_2012 = hourly_summary(june_2012, "cnt")
profile_2012.loc[[8, 13, 17]].round(2)

In [ ]:
profile_2011 = hourly_summary(june_2011, "cnt")

comparison = pd.DataFrame({
    "June 2011": profile_2011["mean"],
    "June 2012": profile_2012["mean"],
})
comparison["increase"] = comparison["June 2012"] - comparison["June 2011"]
comparison.loc[[8, 13, 17]].round(2)

In [ ]:
comparison["increase_pct"] = 100 * comparison["increase"] / comparison["June 2011"]
comparison.loc[[8, 17], ["increase", "increase_pct"]].round(2)

**出力の見所：**返り値は24行の表です。8時と17時では，増加件数と増加率のどちらを比べるかで「大きい」の答えが変わります。関数が共通にしたのは集計規則です。

### 3. 表示と返り値を区別する

次の関数は値を画面に表示します。`shown` に集計表が入るか，実行前に予想します。

In [ ]:
def show_eight(records, value_column):
    table = hourly_summary(records, value_column)
    print(table.loc[8])


shown = show_eight(june_2012, "cnt")
print("Returned None:", shown is None)

In [ ]:
print("Input rows:", len(june_2012))
print("Returned rows:", len(profile_2012))

**出力の見所：**表示されても，`shown` はNoneです。`hourly_summary()` は表を返すので，その後の差の計算や作図に使えます。入力は720行のまま残り，返した集計表が24行になります。

### 4. 同じ入力表で列を替える

登録区分でも朝夕の山は残るでしょうか。呼出にパラメーター名を添え，変えたものが列名であることを確認します。

In [ ]:
registered_2011 = hourly_summary(records=june_2011, value_column="registered")
registered_2012 = hourly_summary(records=june_2012, value_column="registered")

registered_comparison = pd.DataFrame({
    "June 2011": registered_2011["mean"],
    "June 2012": registered_2012["mean"],
})
registered_comparison.loc[[8, 13, 17]].round(2)

In [ ]:
plt.rcParams["figure.figsize"] = (7.0, 3.7)
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, title, first, second in [
    (axes[0], "All rentals", profile_2011, profile_2012),
    (axes[1], "Registered", registered_2011, registered_2012),
]:
    ax.plot(first.index, first["mean"], "o-", markersize=3, label="June 2011")
    ax.plot(second.index, second["mean"], "s--", markersize=3, label="June 2012")
    ax.set(title=title, xlabel="Clock hour", xticks=[0, 6, 12, 18, 23], ylim=(0, 750))
axes[0].set_ylabel("Mean rental count")
axes[1].legend(frameon=False)
plt.show()

**出力の見所：**2012年6月8時の登録区分は472.6件です。全体508.4件との差は未登録区分の平均です。登録区分から利用目的まで直接分かるかは，別に考えます。

### 5. 別の月へ渡して分母に戻る

12月の全体件数を同じ関数へ渡します。朝と夕方の山の高さは，6月と同じ関係でしょうか。

In [ ]:
december_2012 = hourly.loc[
    (hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 12)
]
profile_december = hourly_summary(december_2012, "cnt")
profile_december.loc[[3, 4, 8, 13, 17]].round(2)

In [ ]:
plt.rcParams["figure.figsize"] = (6.6, 3.7)
fig, ax = plt.subplots(layout="constrained")
ax.plot(profile_2012.index, profile_2012["mean"], "o-", markersize=3, label="June 2012")
ax.plot(profile_december.index, profile_december["mean"], "s--", markersize=3, label="December 2012")
ax.set(xlabel="Clock hour", ylabel="Mean rental count", xticks=range(0, 24, 3), ylim=(0, 750))
ax.legend(frameon=False)
plt.show()

In [ ]:
print("Rows in December:", len(december_2012))
print("Dates in December:", december_2012["date"].nunique())
profile_december.loc[profile_december["recorded_hours"] < 31].round(2)

In [ ]:
checks = pd.Series({
    "Rows accounted for": profile_2012["recorded_hours"].sum() == len(june_2012),
    "Rentals accounted for": profile_2012["total"].sum() == june_2012["cnt"].sum(),
    "Eight o'clock matches": profile_2012.loc[8, "mean"] == eight_in_june.mean(),
})
checks

**出力の見所：**12月の3・4時は各30行で，ほかの時刻は31行です。関数が動くことと，分母が同じことは別です。照合のTrueが何を確かめたかを説明し，新しいカーネルからここまで再実行できる手順になっているか読み返します。

## Jupyter 実習（約20分）

ここからは登録・未登録の一方を選び，6月と12月を比べます。関数の最後の行を自分で完成させるため，次の課題セルで関数を定義し直します。講義で作った返り値を流用せず，課題の入力から計算します。

In [ ]:
june = hourly.loc[(hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 6)]
december = hourly.loc[(hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 12)]
print("June rows:", len(june), "December rows:", len(december))

## 1. 比べる区分を選ぶ（約 5 分）

本文では，全体の平均を比べました。2012 年の 6 月では 8 時が 508.4 件，17 時が 680.7 件でした。12 月では約 359.81 件と 375.97 件で，朝夕の高さが近づいていました。

今回は `registered`（登録）か `casual`（未登録）の一方を，結果を見る前に選びます。知りたいことと選んだ理由を一文で書いてください。どちらの区分も，利用目的を直接表す列ではありません。

- 選ぶ区分と理由：
- 6 月と 12 月で朝・昼・夕方のどの違いを調べたいか：

次の二つのセルで，**合計二行**を完成します。関数は入力表を `hr` で集計済みです。まず `return None` を，集計表を返す行へ書き換えます。次に `selected_column = None` を，選んだ列名の文字列へ書き換えます。未記入のままでは後続セルに課題未完と表示されます。

In [ ]:
def hourly_summary(records, value_column):
    grouped = records.groupby("hr")[value_column]
    summary = grouped.agg(recorded_hours="size", total="sum", mean="mean")
    return None  # 課題：集計表 summary を返す行に替える。

In [ ]:
selected_column = None  # 課題："registered" または "casual" を記入する。

## 2. 同じ関数へ二つの表を渡す（約 9 分）

次のセルは，選んだ区分について `hourly_summary(june, selected_column)` と `hourly_summary(december, selected_column)` を呼びます。入力の表は異なり，集計規則は同じです。返された表の 3・4・8・13・17 時を並べます。

実行前に，呼出の `june` と `selected_column` がどのパラメーターへ渡されるかを確認してください。表示を読むときは，`recorded_hours` が 6 月では 30，12 月の一部では 30，ほかでは 31 になることも確かめます。

In [ ]:
june_profile = None
december_profile = None
if selected_column not in ["registered", "casual"]:
    print("課題未完：selected_column に選んだ列名を文字列で指定してください。")
else:
    june_profile = hourly_summary(june, selected_column)
    december_profile = hourly_summary(december, selected_column)
    if june_profile is None or december_profile is None:
        print("課題未完：関数の return で集計表を返してください。")
    else:
        comparison = pd.concat(
            {"June": june_profile, "December": december_profile}, names=["period"]
        )
        display(comparison.loc[(slice(None), [3, 4, 8, 13, 17]), :].round(2))

In [ ]:
if june_profile is None or december_profile is None:
    print("課題未完：返り値の表ができたら，このセルを再実行してください。")
else:
    titles = {"registered": "Registered rentals", "casual": "Casual rentals"}
    fig, ax = plt.subplots(layout="constrained", figsize=(7, 4))
    ax.plot(june_profile.index, june_profile["mean"], "o-", markersize=3, label="June 2012")
    ax.plot(december_profile.index, december_profile["mean"], "s--", markersize=3, label="December 2012")
    upper = max(june_profile["mean"].max(), december_profile["mean"].max()) * 1.1
    ax.set(xlabel="Clock hour", ylabel="Mean rental count", title=titles[selected_column],
           xticks=range(0, 24, 3), ylim=(0, upper))
    ax.legend(frameon=False)
    plt.show()
    print("Highest mean in June:", june_profile["mean"].idxmax(), "h")
    print("Highest mean in December:", december_profile["mean"].idxmax(), "h")

図で一つの山を選び，返り値の行へ戻って，時刻・合計・行数・平均を記録します。6 月と 12 月で高さや位置がどう違うかを一文で説明してください。

12 月の 3・4 時は各 30 行から平均しています。元の表でその時刻がない日を 0 件として追加したものではありません。報告では「収録された時間帯の平均」と分かるようにします。

- 図の一点を作る合計・分母・平均：
- 6 月と 12 月の違い：
- この図だけではまだ分からないこと：

## 3. 再実行して照合する（約 6 分）

ここまでのセルを保存し，カーネル（Colab ではランタイム）の再起動後に，上から全セルを実行します。二行の記入内容はノートに保存されますが，実行時に作った変数と関数は作り直されます。

次のセルで，時刻別に分けた合計を足すと元の全件数に戻るか，行数を失っていないか，8 時の平均が元の行から直接求めた平均と一致するかを確かめます。同じ数字が再び出ることと，その数字が何を表すかの照合を分けて考えます。

In [ ]:
if june_profile is None or december_profile is None:
    print("課題未完：二つの返り値ができてから照合します。")
else:
    checks = []
    for label, records, profile in [
        ("June", june, june_profile), ("December", december, december_profile)
    ]:
        eight = records.loc[records["hr"] == 8, selected_column]
        checks.append({
            "period": label,
            "rows_match": profile["recorded_hours"].sum() == len(records),
            "totals_match": profile["total"].sum() == records[selected_column].sum(),
            "eight_matches": profile.loc[8, "mean"] == eight.mean(),
        })
    display(pd.DataFrame(checks))

次の項目が伝わるよう，図と短表に添える 3〜4 文の報告を書きます。

- 入力データの版，期間，選んだ区分。
- 関数が共通にした規則と，返り値で読んだ時刻・件数の違い。
- 12 月の分母を含めた観測範囲と，利用目的や原因についてまだ言えないこと。
- 最初から再実行できたか，どの出力を元の記録と照合したか。

（ここに報告を記入）

今回の分析は乱数を使っていません。第 10 章の並べ替えのように乱数を使う分析を再現する場合には，追加で何を記録するか一つ挙げてください。

<details>
<summary>二行の記入例と結果の確認</summary>

関数の最後は `return summary`，選択は例えば `selected_column = "registered"` とします。登録区分の時刻別平均は，6 月では 17 時が最大で約 558.23 件，12 月では 8 時が最大で約 350.06 件です。朝夕の高さの関係が変わったことを，図と該当する行で説明できます。登録区分だから全件が通勤とは限りません。

`"casual"` を選んだ場合は，6 月の最大が 17 時の約 122.47 件，12 月の最大が 14 時の約 48.42 件です。答えの最大時刻だけでなく，近い時間帯の形と件数も図で読んでください。

どちらを選んでも12月の3・4時は各30行からの平均です。再実行と照合ができても，欠けた時刻の実際の利用が0だったと証明したことにはなりません。乱数を使う別の分析なら，シードや反復回数も再現条件として残します。

</details>

## 出典

Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository，CC BY 4.0。 [配布元](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) ・ [データ DOI](https://doi.org/10.24432/C5W894)

本書の `bike-sharing-hourly.csv` は UCI 公式 ZIP の `hour.csv` を改名した集計版です。列名・値・行順は同じです。現在の事業者の個票ログとは版が異なり，混ぜて使いません。日付・時計時刻・欠落時刻の性格は Book の出典説明と第 11 章を参照してください。